In [3]:
import pandas as pd
from scipy import stats
from cliffs_delta import cliffs_delta
import numpy as np

# 1. ĐỌC DỮ LIỆU ĐẦU VÀO
try:
    df_bc = pd.read_csv('bc_csr.csv')
    df_ms = pd.read_csv('mutation_scores.csv')
    df = pd.merge(df_bc, df_ms, on='func_id')
except FileNotFoundError:
    print("⚠️ Chưa có data thực tế từ LR (Huy) hoặc chưa chạy xong Mutation.")
    print("Đang mô phỏng dữ liệu với 50 Function IDs thực tế từ dự án...\n")
    
    actual_ids = [
        "PY-005", "PY-006", "PY-007", "PY-008", "PY-009", "PY-020", "PY-023", "PY-025", "PY-028", "PY-011",
        "PY-015", "PY-016", "PY-021", "PY-037", "PY-048", "PY-042", "PY-043", "PY-049", "PY-050", "PY-051",
        "PY-052", "PY-053", "PY-054", "PY-055", "PY-056", "PY-057", "PY-058", "PY-060", "PY-062", "PY-063",
        "PY-064", "PY-065", "PY-066", "PY-067", "PY-068", "PY-069", "PY-070", "PY-071", "PY-072", "PY-073",
        "PY-074", "PY-075", "PY-076", "PY-077", "PY-078", "PY-079", "PY-080", "PY-081", "PY-082", "PY-083"
    ]
    
    np.random.seed(42)
    df = pd.DataFrame({
        'func_id': actual_ids,
        'bc_gpt': np.random.normal(85, 10, 50).clip(0, 100),
        'bc_pynguin': np.random.normal(60, 15, 50).clip(0, 100),
        'ms_gpt': np.random.normal(70, 12, 50).clip(0, 100),
        'ms_pynguin': np.random.normal(45, 20, 50).clip(0, 100),
        'csr_gpt': np.random.choice([1, 0], size=50, p=[0.85, 0.15])
    })

# 2. BONFERRONI CORRECTION
ALPHA = 0.05
NUM_TESTS = 3
ALPHA_ADJ = ALPHA / NUM_TESTS

print(f"Ngưỡng Alpha gốc: {ALPHA}")
print(f"Ngưỡng Alpha hiệu chỉnh (Bonferroni): {ALPHA_ADJ:.4f}\n")
print("=" * 60)

# 3. KIỂM ĐỊNH COMPILATION SUCCESS RATE (CSR)
n_trials = len(df)
n_success = int(df['csr_gpt'].sum())
csr_rate = n_success / n_trials
binom_res = stats.binomtest(k=n_success, n=n_trials, p=0.8, alternative='greater')

print("\n[RQ1 - 1] COMPILATION SUCCESS RATE (CSR)")
print(f"Tỉ lệ CSR: {csr_rate*100:.1f}% ({n_success}/{n_trials})")
print(f"P-value: {binom_res.pvalue:.4e}")
if binom_res.pvalue < ALPHA_ADJ:
    print("✅ Kết luận: Bác bỏ H0. CSR của GPT cao hơn 80% (Có ý nghĩa thống kê).")
else:
    print("❌ Kết luận: Không đủ cơ sở bác bỏ H0 cho CSR.")

# 4. KIỂM ĐỊNH BRANCH COVERAGE (BC)
wilcoxon_bc = stats.wilcoxon(df['bc_gpt'], df['bc_pynguin'], alternative='greater')
delta_bc, res_bc = cliffs_delta(df['bc_gpt'], df['bc_pynguin'])

print("\n[RQ1 - 2] BRANCH COVERAGE (BC)")
print(f"Trung vị GPT: {df['bc_gpt'].median():.1f}% | Pynguin: {df['bc_pynguin'].median():.1f}%")
print(f"P-value: {wilcoxon_bc.pvalue:.4e}")
print(f"Cliff's Delta: {delta_bc:.4f} (Mức độ: {res_bc})")
if wilcoxon_bc.pvalue < ALPHA_ADJ:
    print("✅ Kết luận: Bác bỏ H0. BC của GPT vượt trội hơn Pynguin (Có ý nghĩa thống kê).")
else:
    print("❌ Kết luận: Không đủ cơ sở bác bỏ H0 cho BC.")

# 5. KIỂM ĐỊNH MUTATION SCORE (MS)
wilcoxon_ms = stats.wilcoxon(df['ms_gpt'], df['ms_pynguin'], alternative='greater')
delta_ms, res_ms = cliffs_delta(df['ms_gpt'], df['ms_pynguin'])

print("\n[RQ2] MUTATION SCORE (MS)")
print(f"Trung vị GPT: {df['ms_gpt'].median():.1f}% | Pynguin: {df['ms_pynguin'].median():.1f}%")
print(f"P-value: {wilcoxon_ms.pvalue:.4e}")
print(f"Cliff's Delta: {delta_ms:.4f} (Mức độ: {res_ms})")
if wilcoxon_ms.pvalue < ALPHA_ADJ:
    print("✅ Kết luận: Bác bỏ H0. MS của GPT vượt trội hơn Pynguin (Có ý nghĩa thống kê).")
else:
    print("❌ Kết luận: Không đủ cơ sở bác bỏ H0 cho MS.")
print("\n" + "=" * 60)

⚠️ Chưa có data thực tế từ LR (Huy) hoặc chưa chạy xong Mutation.
Đang mô phỏng dữ liệu với 50 Function IDs thực tế từ dự án...

Ngưỡng Alpha gốc: 0.05
Ngưỡng Alpha hiệu chỉnh (Bonferroni): 0.0167


[RQ1 - 1] COMPILATION SUCCESS RATE (CSR)
Tỉ lệ CSR: 88.0% (44/50)
P-value: 1.0340e-01
❌ Kết luận: Không đủ cơ sở bác bỏ H0 cho CSR.

[RQ1 - 2] BRANCH COVERAGE (BC)
Trung vị GPT: 82.7% | Pynguin: 60.7%
P-value: 3.9702e-13
Cliff's Delta: 0.8488 (Mức độ: large)
✅ Kết luận: Bác bỏ H0. BC của GPT vượt trội hơn Pynguin (Có ý nghĩa thống kê).

[RQ2] MUTATION SCORE (MS)
Trung vị GPT: 70.2% | Pynguin: 48.9%
P-value: 1.1672e-09
Cliff's Delta: 0.7256 (Mức độ: large)
✅ Kết luận: Bác bỏ H0. MS của GPT vượt trội hơn Pynguin (Có ý nghĩa thống kê).

